### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Qwen3_4B_lora_fp16_r256_s15000_i2000_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.95,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            #clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, base_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 05-05 22:41:19 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 05-05 22:41:20 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 05-05 22:41:25 [config.py:585] This model supports multiple tasks: {'score', 'generate', 'classify', 'embed', 'reward'}. Defaulting to 'generate'.
INFO 05-05 22:41:25 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-05 22:41:26 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Qwen3_4B_lora_fp16_r256_s15000_i2000_msl2048', speculative_config=None, tokenizer='./lora/Qwen3_4B_lora_fp16_r256_s15000_i2000_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_de

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-05 22:41:30 [loader.py:447] Loading weights took 2.65 seconds
INFO 05-05 22:41:30 [gpu_model_runner.py:1186] Model loading took 7.5454 GB and 2.950441 seconds
INFO 05-05 22:41:32 [kv_cache_utils.py:566] GPU KV cache size: 8,144 tokens
INFO 05-05 22:41:32 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 7.95x
INFO 05-05 22:41:37 [gpu_model_runner.py:1534] Graph capturing finished in 5 secs, took 0.08 GiB
INFO 05-05 22:41:37 [core.py:151] init engine (profile, create kv cache, warmup model) took 7.05 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
import sys
sys.path.append(r'/home/vino/ML_Projects/Drawing_with_LLMs/drawing-with-llms')
import pandas as pd

df1=pd.read_csv(r'./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv',header=[0])
df2=pd.read_csv(r'./drawing-with-llms/test_filtered_2_batch_vqa_gemini_2o_kaggle.csv',header=[0])
#df3=pd.read_csv(r'./drawing-with-llms/gemini_25_pro_validation/train_filtered_1_batch_gpt4.csv',header=[0])
#print(df3.shape)
#df2=df2.drop_duplicates(['description'])
#df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df1.drop_duplicates(['description'])

print(df.shape)
df.head(2)

(71, 7)


,description,clean_svg,sl_score,response,vqa_pair,response_2,gpt_svg_2
0,"'Vibrant autumn forest',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.904117,Here is the visual question answering (VQA) pa...,"{'description': 'Vibrant autumn forest', 'ques...","Here's an improved SVG representation of a ""Vi...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,"'Morning dew on grass',","<svg viewBox=""0 0 200 200"" width=""200"" height=...",0.987024,Here is a visual question answering (VQA) pair...,"{'description': 'Morning dew on grass', 'quest...","Here's an improved SVG representation of ""Morn...","<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."


In [7]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [8]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 15
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                   | 0/5 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:10<02:22, 10.19s/it, est. speed input: 5.79 
cessed prompts:  13%|▏| 2/15 [00:10<01:00,  4.62s/it, est. speed input: 10.82
cessed prompts:  20%|▏| 3/15 [00:12<00:36,  3.08s/it, est. speed input: 14.64
cessed prompts:  27%|▎| 4/15 [00:13<00:25,  2.28s/it, est. speed input: 18.02
cessed prompts:  33%|▎| 5/15 [00:14<00:19,  1.97s/it, est. speed input: 20.38
cessed prompts:  40%|▍| 6/15 [00:16<00:17,  1.98s/it, est. speed input: 21.66
cessed prompts:  47%|▍| 7/15 [00:17<00:13,  1.67s/it, est. speed input: 23.88
cessed prompts:  53%|▌| 8/15 [00:18<00:09,  1.34s/it, est. speed input: 26.36
cessed prompts:  60%|▌| 9/15 [00:18<00:05,  1.03it/s, est. speed input: 29.36
cessed prompts:  67%|▋| 10/15 [00:19<00:04,  1.02it/s, est. speed input: 30.9
cessed prompts:  73%|▋| 11/15 [00:21<00:05,  1.26s/it, est. s

Failed to convert Windy wheat fields due to not well-formed (invalid token): line 1, column 1786, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:06<01:25,  6.07s/it, est. speed input: 10.37
cessed prompts:  13%|▏| 2/15 [00:09<00:55,  4.27s/it, est. speed input: 14.00
cessed prompts:  20%|▏| 3/15 [00:11<00:42,  3.52s/it, est. speed input: 15.98
cessed prompts:  27%|▎| 4/15 [00:13<00:29,  2.64s/it, est. speed input: 19.38
cessed prompts:  33%|▎| 5/15 [00:14<00:21,  2.12s/it, est. speed input: 22.19
cessed prompts:  40%|▍| 6/15 [00:14<00:13,  1.50s/it, est. speed input: 26.17
cessed prompts:  47%|▍| 7/15 [00:15<00:11,  1.41s/it, est. speed input: 28.13
cessed prompts:  53%|▌| 8/15 [00:16<00:07,  1.09s/it, est. speed input: 31.47
cessed prompts:  60%|▌| 9/15 [00:17<00:06,  1.03s/it, est. speed input: 33.69
cessed prompts:  67%|▋| 10/15 [00:20<00:08,  1.71s/it, est. speed input: 31.4
cessed prompts:  73%|▋| 11/15 [00:21<00:06,  1.68s/it, est. speed input: 32.1
cessed prompts:  80%|▊| 12/15 [00:23<00:05,  1.70s/it, est. spe

Failed to convert Winter landscape with snow-covered trees due to not well-formed (invalid token): line 26, column 43, Returning default SVG.
Failed to convert Forest pathway due to not well-formed (invalid token): line 50, column 52, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:05<01:13,  5.23s/it, est. speed input: 12.99
cessed prompts:  13%|▏| 2/15 [00:09<01:03,  4.92s/it, est. speed input: 12.78
cessed prompts:  20%|▏| 3/15 [00:11<00:39,  3.28s/it, est. speed input: 16.97
cessed prompts:  27%|▎| 4/15 [00:12<00:25,  2.35s/it, est. speed input: 20.61
cessed prompts:  33%|▎| 5/15 [00:13<00:18,  1.90s/it, est. speed input: 23.58
cessed prompts:  40%|▍| 6/15 [00:14<00:14,  1.56s/it, est. speed input: 26.22
cessed prompts:  47%|▍| 7/15 [00:14<00:09,  1.17s/it, est. speed input: 29.83
cessed prompts:  53%|▌| 8/15 [00:14<00:06,  1.09it/s, est. speed input: 33.46
cessed prompts:  60%|▌| 9/15 [00:19<00:12,  2.15s/it, est. speed input: 28.49
cessed prompts:  67%|▋| 10/15 [00:20<00:07,  1.57s/it, est. speed input: 31.0
cessed prompts:  73%|▋| 11/15 [00:25<00:10,  2.65s/it, est. speed input: 27.3
cessed prompts:  87%|▊| 13/15 [00:32<00:06,  3.06s/it, est. spe

Failed to convert Buildings lit up as the sun sets in the city. due to not well-formed (invalid token): line 1, column 1973, Returning default SVG.



cessed prompts:   0%| | 0/11 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   9%| | 1/11 [00:10<01:46, 10.65s/it, est. speed input: 5.82 
cessed prompts:  18%|▏| 2/11 [00:12<00:51,  5.70s/it, est. speed input: 9.63 
cessed prompts:  27%|▎| 3/11 [00:14<00:29,  3.69s/it, est. speed input: 13.05
cessed prompts:  45%|▍| 5/11 [00:15<00:12,  2.11s/it, est. speed input: 19.21
cessed prompts:  55%|▌| 6/11 [00:16<00:08,  1.71s/it, est. speed input: 22.09
cessed prompts:  64%|▋| 7/11 [00:16<00:05,  1.26s/it, est. speed input: 25.58
cessed prompts:  73%|▋| 8/11 [00:17<00:02,  1.05it/s, est. speed input: 29.12
cessed prompts:  82%|▊| 9/11 [00:19<00:02,  1.34s/it, est. speed input: 28.90
cessed prompts:  91%|▉| 10/11 [00:21<00:01,  1.68s/it, est. speed input: 28.4
Processed prompts: 100%|█| 11/11 [00:29<00:00,  2.67s/it, est. speed input: 23.2
Batch prediction: 100%|███████████████████████████| 5/5 [02:48<00:00, 33.62s/it]


In [9]:
df['svg_3']=results

/tmp/ipykernel_66001/1893269863.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['svg_3']=results


In [10]:
model.close_model()

In [11]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
100%|███████████████████████████████████████████| 71/71 [00:04<00:00, 16.15it/s]
/tmp/ipykernel_66001/1861074108.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|███████████████████████████████████████████| 71/71 [00:07<00:00,  9.13it/s]
/tmp/ipykernel_66001/2425145936.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)


In [14]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

/tmp/ipykernel_66001/1028116435.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3


In [15]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.6110425568869432 mean_aes_score: 0.44789651481198595 combined_score: 0.5566605428619574


In [16]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 4
default_svg_score_mean: 6.042528459364316e-08 default_aes_score_mean: 0.43699469566345217 combined_score: 0.1456649388380071


In [17]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 67
non-default_svg_score_mean: 0.6475227059294303 non-default_aes_score_mean: 0.4485473696865253 combined_score: 0.5811975938484619
